# Review Authenticity Engine

A complete training notebook for the **Review Authenticity Engine**.

**Live demo:** https://review-trust-engine.streamlit.app/

---

**Overview**

- DistilBERT review classifier
- Synthetic fake review generation
- Leak-safe train/test split
- Reviewer graph construction
- Louvain community detection
- Export for the Streamlit application

---

**Environment**

- Python 3.11
- Packages in `requirements.txt`
- Trained on Kaggle (dual NVIDIA T4 GPUs)


## Environment

- Python 3.11
- Packages are listed in `requirements.txt`
- Hardware used: Kaggle Notebooks, dual NVIDIA T4 GPUs

## Loading the base data

Real Amazon Reviews subset (McAuley Lab 2023, Electronics category) — this is the "genuine review" population fake stuff gets mixed into.

In [ ]:
# Update this path to where you stored the dataset locally
DATA_DIR = "data"

In [ ]:
import pandas as pd

df = pd.read_parquet(f"{DATA_DIR}/fraudscope_real_reviews.parquet")
print(df.shape)

## Generating the fake reviews

Three tiers, grouped into "rings" — a set of fake accounts hitting the same product in a time window:

- **Tier 1** — obvious template spam
- **Tier 2** — combinatorial templates, a bit more variety
- **Tier 3** — paraphrased with Pegasus so it reads naturally, posted with staggered timing

In [ ]:
import random
import numpy as np
import pandas as pd
from datetime import timedelta

def make_fake_uid(i):
    return f"FAKEUSER{i:05d}"

tier1_openers = ["Great", "Amazing", "Excellent", "Perfect", "Best", "Awesome", "Fantastic", "Incredible"]
tier1_nouns = ["product", "item", "purchase", "buy", "deal"]
tier1_closers = [
    "highly recommend buy it now",
    "will buy again for sure",
    "very satisfied five stars",
    "best purchase ever made",
    "works great no complaints",
    "totally worth the money",
    "five stars all the way",
    "everyone should get this"
]

def make_tier1_text():
    return f"{random.choice(tier1_openers)} {random.choice(tier1_nouns)}, {random.choice(tier1_closers)}"

tier2_adjectives = ["amazing", "fantastic", "incredible", "superb", "outstanding", "wonderful", "impressive"]
tier2_nouns = ["product", "item", "purchase", "gadget", "device"]
tier2_reasons = ["fast shipping", "great build quality", "perfect fit", "excellent value", "works flawlessly"]

def make_tier2_text():
    return f"This is a {random.choice(tier2_adjectives)} {random.choice(tier2_nouns)}. {random.choice(tier2_reasons).capitalize()} and {random.choice(tier2_reasons)}. Would definitely recommend to anyone looking for this."

def make_tier3_text():
    return random.choice(paraphrased_pool)

def generate_ring(asin, n_reviewers, tier, base_time):
    rows = []
    reviewer_ids = [make_fake_uid(random.randint(1, 999999)) for _ in range(n_reviewers)]
    for idx, uid in enumerate(reviewer_ids):
        if tier == 1:
            text, rating, ts = make_tier1_text(), 5.0, base_time + timedelta(minutes=random.randint(0, 30))
        elif tier == 2:
            text, rating, ts = make_tier2_text(), random.choice([4.0, 5.0]), base_time + timedelta(hours=random.randint(0, 72))
        else:
            text, rating = make_tier3_text(), random.choice([4.0, 4.0, 5.0])
            ts = base_time + timedelta(hours=random.randint(6, 72) * (idx + 1) / n_reviewers)
        rows.append({"rating": rating, "title": "", "text": text, "asin": asin, "user_id": uid,
                      "timestamp": ts, "verified_purchase": False, "label": 1, "tier": tier})
    return rows

print("helpers defined")

## Paraphrasing for tier 3

Pegasus over 600 varied seed sentences pulled from real reviews. This variety is what actually fixed the leak — the first version I ran only had about 20 fixed sentences here, which was the whole problem.

> **Note:** the pip install below is Kaggle-specific setup, not part of the deployed app — see `requirements.txt` for what the Streamlit app itself needs.

In [ ]:
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
para_tokenizer = PegasusTokenizer.from_pretrained("tuner007/pegasus_paraphrase")
para_model = PegasusForConditionalGeneration.from_pretrained("tuner007/pegasus_paraphrase").to(device)

def paraphrase_batch(texts, max_length=60):
    batch = para_tokenizer(texts, truncation=True, padding="longest", max_length=max_length, return_tensors="pt").to(device)
    translated = para_model.generate(**batch, max_length=max_length, num_beams=5, num_return_sequences=1)
    return para_tokenizer.batch_decode(translated, skip_special_tokens=True)

seed_pool = df[(df["text"].str.len() > 40) & (df["text"].str.len() < 200)]["text"].sample(n=600, random_state=42).tolist()

paraphrased_pool = []
batch_size = 32
for i in range(0, len(seed_pool), batch_size):
    paraphrased_pool.extend(paraphrase_batch(seed_pool[i:i+batch_size]))
    if i % 160 == 0:
        print(f"{i}/{len(seed_pool)}")

print(f"generated {len(paraphrased_pool)} paraphrased seeds")

import json
with open("/kaggle/working/paraphrased_pool.json", "w") as f:
    json.dump(paraphrased_pool, f)
print("saved to /kaggle/working/paraphrased_pool.json")

## Putting it all together

400 products get a fake ring, spread roughly evenly across the three tiers, mixed in with the real reviews.

In [ ]:
random.seed(42)
target_asins = df["asin"].drop_duplicates().sample(n=400, random_state=42).tolist()
fake_rows = []
base_start = pd.Timestamp("2022-06-01")
for i, asin in enumerate(target_asins):
    tier = [1, 1, 2, 2, 3, 3, 3][i % 7]
    n_reviewers = random.randint(8, 20)
    base_time = base_start + timedelta(days=random.randint(0, 400))
    fake_rows.extend(generate_ring(asin, n_reviewers, tier, base_time))

fake_df = pd.DataFrame(fake_rows)
df2 = df.copy()
df2["label"] = 0; df2["tier"] = 0
full_df = pd.concat([df2, fake_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
full_df["text"] = full_df["title"].fillna("") + " " + full_df["text"].fillna("")

print(full_df.shape)
print(full_df["tier"].value_counts())
print("unique tier-1 texts:", full_df[full_df["tier"]==1]["text"].nunique(), "out of", len(full_df[full_df["tier"]==1]))
print("unique tier-3 texts:", full_df[full_df["tier"]==3]["text"].nunique(), "out of", len(full_df[full_df["tier"]==3]))

full_df.to_parquet("/kaggle/working/fraudscope_full_labeled_v2.parquet", index=False)

## Splitting train/test (the leak-safe way)

Splitting by unique text instead of by row means no review can end up in both train and test. This is the actual fix for the leak.

In [ ]:
import numpy as np

np.random.seed(42)

def text_based_split(full_df, test_size=0.2):
    unique_texts = full_df["text"].unique()
    np.random.shuffle(unique_texts)
    n_test = int(len(unique_texts) * test_size)
    test_texts = set(unique_texts[:n_test])
    is_test = full_df["text"].isin(test_texts)
    return full_df[~is_test].reset_index(drop=True), full_df[is_test].reset_index(drop=True)

train_df, test_df = text_based_split(full_df, test_size=0.2)

print(train_df.shape, test_df.shape)
print(train_df["tier"].value_counts())
print(test_df["tier"].value_counts())

overlap_check = set(train_df["text"]) & set(test_df["text"])
print("text overlap between train/test:", len(overlap_check))

train_df.to_parquet("/kaggle/working/fraudscope_train_v2.parquet", index=False)
test_df.to_parquet("/kaggle/working/fraudscope_test_v2.parquet", index=False)

## Training the classifier

The actual final run, on the leak-free split (checkpoint dir says v5 just because it was the fifth attempt after all the debugging).

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
import evaluate

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_df[["text", "label"]])
test_ds = Dataset.from_pandas(test_df[["text", "label"]])
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
        "precision": precision_metric.compute(predictions=preds, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/fraudscope_model_v5_checkpoints",
    eval_strategy="epoch", save_strategy="epoch",
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, report_to="none", fp16=True
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()

trainer.save_model("/kaggle/working/review_authenticity_model_v5")
tokenizer.save_pretrained("/kaggle/working/review_authenticity_model_v5")

## A dead end I tried along the way

Before I figured out the real problem, I tried weighting the loss function to deal with class imbalance. Made recall worse, not better, so I dropped it.

Turned out the actual fix was giving tier-1 way more template variety, so the leak-safe split still had something representative to learn from on both sides — not a modeling trick, just better data.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_df["label"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to("cuda")
print("class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="/kaggle/working/fraudscope_model_v4_checkpoints",
    eval_strategy="epoch", save_strategy="epoch",
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, report_to="none", fp16=True
)

trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()

trainer.save_model("/kaggle/working/review_authenticity_model_v4")
tokenizer.save_pretrained("/kaggle/working/review_authenticity_model_v4")

## Checking it actually generalizes, per tier

The real test — not overall accuracy, but does it hold up tier by tier on data it's never seen.

In [ ]:
import torch

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict_batch(texts, batch_size=64):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
    return preds

test_df = test_df.reset_index(drop=True)
test_df["pred"] = predict_batch(test_df["text"].tolist())

for tier in sorted(test_df["tier"].unique()):
    sub = test_df[test_df["tier"] == tier]
    if tier == 0:
        acc = (sub["pred"] == 0).mean()
        print(f"tier {tier} (real): correctly kept as real = {acc:.3f}  (n={len(sub)})")
    else:
        recall = (sub["pred"] == 1).mean()
        print(f"tier {tier}: recall (caught as fake) = {recall:.3f}  (n={len(sub)})")

## Building the reviewer graph

Weighted reviewer-to-reviewer graph based on three things: how close in time two people reviewed the same product, how similar their text is (MiniLM embeddings), and how many products they both touched. Then Louvain to find dense clusters — candidate rings.

> **Note:** the pip install below is Kaggle-specific setup, not part of the deployed app — see `requirements.txt` for what the Streamlit app itself needs.

In [ ]:
full_df = pd.read_parquet(f"{DATA_DIR}/fraudscope_full_labeled_v2.parquet")
print(full_df.shape)

In [ ]:
import numpy as np
import networkx as nx
import community.community_louvain as community_louvain
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

embedder = SentenceTransformer("all-MiniLM-L6-v2")

reviewer_products = full_df.groupby("user_id")["asin"].apply(set).to_dict()
product_reviewers = full_df.groupby("asin")["user_id"].apply(list).to_dict()

candidate_pairs = set()
for asin, reviewers in product_reviewers.items():
    uniq = set(reviewers)
    if len(uniq) < 2 or len(uniq) > 60:
        continue
    for a, b in combinations(uniq, 2):
        candidate_pairs.add((a, b) if a < b else (b, a))

print(f"candidate reviewer pairs: {len(candidate_pairs)}")

involved_users = set(u for pair in candidate_pairs for u in pair)
sub_df = full_df[full_df["user_id"].isin(involved_users)].copy()
sub_df["embedding"] = list(embedder.encode(sub_df["text"].tolist(), show_progress_bar=True, batch_size=128))

user_embeddings = sub_df.groupby("user_id")["embedding"].apply(lambda x: np.mean(np.stack(x), axis=0)).to_dict()
user_asin_times = full_df.groupby(["user_id", "asin"])["timestamp"].apply(list).to_dict()

def timing_score(u, v, asin):
    tu = user_asin_times.get((u, asin))
    tv = user_asin_times.get((v, asin))
    if not tu or not tv:
        return 0
    min_gap = min(abs((pd.Timestamp(a) - pd.Timestamp(b)).total_seconds()) for a in tu for b in tv)
    return np.exp(-min_gap / (3600 * 24))

G = nx.Graph()
for u, v in candidate_pairs:
    shared = reviewer_products[u] & reviewer_products[v]
    if not shared:
        continue
    overlap_score = len(shared) / min(len(reviewer_products[u]), len(reviewer_products[v]))
    text_sim = cosine_similarity([user_embeddings[u]], [user_embeddings[v]])[0][0]
    time_score = max(timing_score(u, v, asin) for asin in shared)
    weight = 0.4 * time_score + 0.3 * text_sim + 0.3 * overlap_score
    if weight > 0.15:
        G.add_edge(u, v, weight=float(weight))

partition = community_louvain.best_partition(G, weight="weight")
print(f"graph nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

## Checking the clusters against ground truth

Since I know which reviewer IDs are fake (they're all prefixed `FAKEUSER`), I can check directly how well the graph clusters line up with the rings I actually planted.

In [ ]:
partition_df = pd.DataFrame(list(partition.items()), columns=["user_id", "cluster"])
partition_df["is_fake"] = partition_df["user_id"].str.startswith("FAKEUSER")

cluster_stats = partition_df.groupby("cluster").agg(
    size=("user_id", "count"),
    fake_count=("is_fake", "sum")
).reset_index()
cluster_stats["fake_ratio"] = cluster_stats["fake_count"] / cluster_stats["size"]

small_dense_clusters = cluster_stats[(cluster_stats["size"] >= 4) & (cluster_stats["size"] <= 30)]
high_purity = small_dense_clusters[small_dense_clusters["fake_ratio"] > 0.5]
print(f"high purity clusters: {len(high_purity)}")

## Exporting for the dashboard

This JSON is what the "Detected Fraud Rings" tab in the Streamlit app actually reads from — nothing gets recomputed live, it's just this export.

In [ ]:
import json

ring_export = []
for _, row in high_purity.iterrows():
    cluster_id = row["cluster"]
    members = partition_df[partition_df["cluster"] == cluster_id]["user_id"].tolist()
    member_reviews = full_df[full_df["user_id"].isin(members)][["user_id", "asin", "text", "tier", "rating"]].to_dict("records")
    ring_export.append({
        "cluster_id": int(cluster_id),
        "size": int(row["size"]),
        "fake_ratio": float(row["fake_ratio"]),
        "reviews": member_reviews[:20]
    })

with open("/kaggle/working/detected_rings.json", "w") as f:
    json.dump(ring_export, f, indent=2)
print(f"exported {len(ring_export)} rings")

## Conclusion

This notebook covers the full pipeline behind the Review Authenticity Engine, start to finish:

Dataset → synthetic fraud generation → leak-safe split → DistilBERT fine-tuning → per-tier evaluation → reviewer graph → Louvain detection → export to dashboard

The deployed app loads the trained model from Hugging Face directly, and the "Detected Fraud Rings" tab reads from the JSON exported at the end of this notebook — neither of those steps get recomputed live in the app itself.